# 손상 차량 부품 분류 — Colab GPU 학습

손상 부위 crop이 어느 차량 부품인지 10개 그룹(앞범퍼, 앞펜더, 전조등, 뒷범퍼, 뒷펜더, 후미등, 옆면, 사이드미러, 스텝, 타이어/휠) 중 하나로 분류하는 EfficientNet-B3 모델을 학습합니다. 손상 위치 자체를 찾는 모델이 아니라, 기존 손상 세그멘테이션 결과의 crop을 입력으로 받는 분류 모델입니다.

## 실행 방법
1. 서브 Google 계정으로 Colab에 로그인하고 런타임 유형을 GPU로 설정합니다.
2. 검증된 `train/코랩_부품.zip`을 서브 계정 Drive의 `MyDrive/car_damage_part_classifier/` 폴더에 업로드합니다. 이 ZIP은 최상위에 `effb3_damage_part_full_crops/` 폴더가 하나 들어 있는 구조입니다.
3. 아래 셀을 순서대로 실행합니다. 전체 데이터에서 라벨을 붙일 수 있는 crop을 모두 포함합니다: train 341,088장, validation 60,095장(총 401,183장). ImageNet 사전학습 가중치로 최대 5 epoch 학습하고 validation Macro F1 조기 종료를 사용합니다.
4. 체크포인트와 학습 기록은 Drive의 `car_damage_part_classifier/runs/`에 저장됩니다.

Google Drive 무료 용량과 Colab GPU 가용성은 계정 설정에 따라 다를 수 있습니다. JPEG crop 전체를 Bandizip 저장 압축으로 묶어 10GB 이하 ZIP으로 만듭니다.

In [ ]:
from datetime import datetime
from pathlib import Path
import shutil
import zipfile

from google.colab import drive
drive.mount("/content/drive")

DRIVE_ROOT = Path("/content/drive/MyDrive/car_damage_part_classifier")
DATASET_ZIPS = [DRIVE_ROOT / "코랩_부품.zip", DRIVE_ROOT / "car_damage_part_crops_10gb.zip"]
DATASET_ZIP = next((path for path in DATASET_ZIPS if path.is_file()), DATASET_ZIPS[0])
DATASET_ROOT = Path("/content/effb3_part_dataset")
LOCAL_DATASET_ZIP = Path("/content") / DATASET_ZIP.name
DRIVE_RUN_ROOT = DRIVE_ROOT / "runs"
RUN_NAME = f"effb3_part_full_{datetime.now():%Y%m%d_%H%M%S}"
RUN_DIR = DRIVE_RUN_ROOT / RUN_NAME
LAST_CHECKPOINT = RUN_DIR / "last.pt"

EPOCHS = 5
EARLY_STOPPING_PATIENCE = 2
BATCH_SIZE = 16
IMAGE_SIZE = 300
NUM_WORKERS = 2
RESUME = False

if not DATASET_ZIP.is_file():
    raise FileNotFoundError(f"Drive에 데이터 ZIP을 업로드하세요: {DATASET_ZIP}")
if RESUME and not LAST_CHECKPOINT.is_file():
    raise FileNotFoundError(f"이어 학습할 체크포인트가 없습니다: {LAST_CHECKPOINT}")
if not RESUME and RUN_DIR.exists():
    raise FileExistsError(f"실행 폴더가 이미 있습니다: {RUN_DIR}. 새 RUN_NAME을 사용하세요.")
print(f"Dataset ZIP: {DATASET_ZIP} ({DATASET_ZIP.stat().st_size / 1e9:.2f} GB)")
print(f"Run output: {RUN_DIR}")

In [ ]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError("Colab 런타임 유형을 GPU로 변경한 뒤 다시 실행하세요.")
device = torch.device("cuda")
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"PyTorch: {torch.__version__} | CUDA: {torch.version.cuda}")

required_free = DATASET_ZIP.stat().st_size * 3
free_bytes = shutil.disk_usage("/content").free
if free_bytes < required_free:
    raise OSError(
        f"Colab 런타임 디스크 여유가 부족합니다: {free_bytes / 1e9:.1f} GB; "
        f"압축 해제에 최소 약 {required_free / 1e9:.1f} GB 필요"
    )
if not LOCAL_DATASET_ZIP.is_file():
    print("Drive → Colab 로컬 디스크 복사 중...")
    shutil.copy2(DATASET_ZIP, LOCAL_DATASET_ZIP)
if DATASET_ROOT.exists():
    manifest_path = DATASET_ROOT / "effb3_damage_part_full_crops" / "dataset_manifest.json"
    if not manifest_path.is_file():
        raise FileExistsError(f"불완전한 데이터 폴더가 있습니다: {DATASET_ROOT}. 런타임을 초기화하세요.")
    print(f"기존 압축 해제 데이터 재사용: {DATASET_ROOT}")
else:
    DATASET_ROOT.mkdir(parents=True)
    with zipfile.ZipFile(LOCAL_DATASET_ZIP) as archive:
        bad_member = archive.testzip()
        if bad_member is not None:
            raise zipfile.BadZipFile(f"ZIP 손상 파일: {bad_member}")
        archive.extractall(DATASET_ROOT)
    print(f"데이터 압축 해제 완료: {DATASET_ROOT}")

CROP_ROOT = DATASET_ROOT / "effb3_damage_part_full_crops"
LOCAL_RUN_ROOT = Path("/content/effb3_runs")
CLASS_NAMES = ["앞범퍼", "앞펜더", "전조등", "뒷범퍼", "뒷펜더", "후미등", "옆면", "사이드미러", "스텝", "타이어/휠"]
class_to_dir = {name: f"{index:02d}_{name.replace('/', '-')}" for index, name in enumerate(CLASS_NAMES)}
directory_to_class = {directory: name for name, directory in class_to_dir.items()}
manifest = json.loads((CROP_ROOT / "dataset_manifest.json").read_text(encoding="utf-8"))
if manifest.get("class_names") != CLASS_NAMES or manifest.get("train_crops") != 341088 or manifest.get("val_crops") != 60095:
    raise ValueError(f"10GB 전체 데이터 ZIP이 아닙니다. manifest 확인 필요: {manifest}")

In [ ]:
import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets, models, transforms

weights = models.EfficientNet_B3_Weights.DEFAULT
mean, std = weights.transforms().mean, weights.transforms().std
train_transform = transforms.Compose([
    transforms.RandomResizedCrop(IMAGE_SIZE, scale=(0.85, 1.0), ratio=(0.9, 1.1)),
    # 좌우 반전은 L/R 부품 라벨을 바꾸므로 적용하지 않습니다.
    transforms.ColorJitter(brightness=0.15, contrast=0.15, saturation=0.1),
    transforms.ToTensor(),
    transforms.Normalize(mean, std),
])
val_transform = transforms.Compose([
    transforms.Resize(320),
    transforms.CenterCrop(IMAGE_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(mean, std),
])
train_dataset = datasets.ImageFolder(CROP_ROOT / "train", transform=train_transform)
val_dataset = datasets.ImageFolder(CROP_ROOT / "val", transform=val_transform)
expected_dirs = [class_to_dir[name] for name in CLASS_NAMES]
if train_dataset.classes != expected_dirs or val_dataset.classes != expected_dirs:
    raise ValueError(f"데이터의 클래스 폴더가 예상과 다릅니다: {train_dataset.classes}")
part_names = [directory_to_class[name] for name in train_dataset.classes]
train_counts = torch.bincount(torch.tensor(train_dataset.targets), minlength=len(part_names)).float()
val_counts = torch.bincount(torch.tensor(val_dataset.targets), minlength=len(part_names))
if torch.any(train_counts == 0) or torch.any(val_counts == 0):
    raise ValueError("각 클래스에 train/validation crop이 모두 있어야 합니다.")

train_loader = DataLoader(
    train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS,
    pin_memory=True, persistent_workers=NUM_WORKERS > 0,
)
val_loader = DataLoader(
    val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS,
    pin_memory=True, persistent_workers=NUM_WORKERS > 0,
)
print(f"Part classes ({len(part_names)}): {part_names}")
print(f"Train crops: {len(train_dataset):,} | Validation crops: {len(val_dataset):,}")
print("Train class counts:", {name: int(train_counts[i]) for i, name in enumerate(part_names)})

In [ ]:
import csv
import time
from sklearn.metrics import accuracy_score, f1_score, classification_report

RUN_DIR = DRIVE_RUN_ROOT / RUN_NAME
LAST_CHECKPOINT = RUN_DIR / "last.pt"
BEST_CHECKPOINT = RUN_DIR / "best.pt"
HISTORY_CSV = RUN_DIR / "history.csv"
model = models.efficientnet_b3(weights=weights)
feature_count = model.classifier[1].in_features
model.classifier[1] = nn.Linear(feature_count, len(part_names))
model = model.to(device)

class_weights = train_counts.clamp_min(1).reciprocal().sqrt()
class_weights = class_weights / class_weights.mean()
class_weights = class_weights.clamp(max=5.0).to(device)
criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="max", factor=0.5, patience=1,
)
scaler = torch.amp.GradScaler("cuda", enabled=device.type == "cuda")
start_epoch = 0
best_macro_f1 = 0.0
epochs_without_improvement = 0

if RESUME:
    if not LAST_CHECKPOINT.is_file():
        raise FileNotFoundError(f"이어 학습할 last.pt가 없습니다: {LAST_CHECKPOINT}")
    checkpoint = torch.load(LAST_CHECKPOINT, map_location=device, weights_only=False)
    if checkpoint.get("class_names") != part_names:
        raise ValueError("체크포인트의 클래스와 현재 데이터 클래스가 다릅니다.")
    model.load_state_dict(checkpoint["model"])
    optimizer.load_state_dict(checkpoint["optimizer"])
    scheduler.load_state_dict(checkpoint["scheduler"])
    scaler.load_state_dict(checkpoint["scaler"])
    start_epoch = checkpoint["epoch"]
    best_macro_f1 = checkpoint["best_macro_f1"]
    epochs_without_improvement = checkpoint.get("epochs_without_improvement", 0)
    print(f"Epoch {start_epoch}까지 저장된 체크포인트에서 이어갑니다.")
elif RUN_DIR.exists():
    raise FileExistsError(
        f"실행 폴더가 이미 있습니다: {RUN_DIR}. RESUME=True로 이어가거나 RUN_NAME을 바꾸세요."
    )

RUN_DIR.mkdir(parents=True, exist_ok=True)
print(f"EfficientNet-B3 | classes={len(part_names)} | device={device} | run={RUN_DIR}")

In [ ]:
from tqdm.auto import tqdm

def run_epoch(loader, training):
    model.train(training)
    total_loss = 0.0
    all_targets, all_predictions = [], []
    progress_bar = tqdm(loader, leave=False, desc="train" if training else "validation")
    for images, targets in progress_bar:
        images = images.to(device, non_blocking=True)
        targets = targets.to(device, non_blocking=True)
        if training:
            optimizer.zero_grad(set_to_none=True)
        with torch.set_grad_enabled(training):
            with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=device.type == "cuda"):
                logits = model(images)
                loss = criterion(logits, targets)
            if training:
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()
        total_loss += loss.item() * targets.size(0)
        predictions = logits.argmax(dim=1)
        all_targets.extend(targets.cpu().tolist())
        all_predictions.extend(predictions.cpu().tolist())
        progress_bar.set_postfix(loss=f"{loss.item():.3f}")
    average_loss = total_loss / len(loader.dataset)
    accuracy = accuracy_score(all_targets, all_predictions)
    macro_f1 = f1_score(all_targets, all_predictions, average="macro", zero_division=0)
    return average_loss, accuracy, macro_f1, all_targets, all_predictions

for epoch in range(start_epoch, EPOCHS):
    epoch_started = time.monotonic()
    train_loss, train_accuracy, train_f1, _, _ = run_epoch(train_loader, training=True)
    val_loss, val_accuracy, val_f1, val_targets, val_predictions = run_epoch(val_loader, training=False)
    scheduler.step(val_f1)
    improved = val_f1 > best_macro_f1
    if improved:
        best_macro_f1 = val_f1
        epochs_without_improvement = 0
    else:
        epochs_without_improvement += 1
    epoch_number = epoch + 1
    row = {
        "epoch": epoch_number,
        "train_loss": train_loss,
        "train_accuracy": train_accuracy,
        "train_macro_f1": train_f1,
        "val_loss": val_loss,
        "val_accuracy": val_accuracy,
        "val_macro_f1": val_f1,
        "seconds": time.monotonic() - epoch_started,
    }
    print(
        f"Epoch {epoch_number}/{EPOCHS} | "
        f"train loss {train_loss:.4f}, acc {train_accuracy:.4f}, macro-F1 {train_f1:.4f} | "
        f"val loss {val_loss:.4f}, acc {val_accuracy:.4f}, macro-F1 {val_f1:.4f} | "
        f"{row['seconds'] / 60:.1f} min"
    )
    with HISTORY_CSV.open("a", newline="", encoding="utf-8") as file:
        writer = csv.DictWriter(file, fieldnames=row.keys())
        if file.tell() == 0:
            writer.writeheader()
        writer.writerow(row)

    checkpoint = {
        "epoch": epoch_number,
        "model": model.state_dict(),
        "optimizer": optimizer.state_dict(),
        "scheduler": scheduler.state_dict(),
        "scaler": scaler.state_dict(),
        "class_names": part_names,
        "best_macro_f1": best_macro_f1,
        "epochs_without_improvement": epochs_without_improvement,
        "image_size": IMAGE_SIZE,
        "architecture": "efficientnet_b3",
    }
    temporary_checkpoint = RUN_DIR / "last.tmp.pt"
    torch.save(checkpoint, temporary_checkpoint)
    temporary_checkpoint.replace(LAST_CHECKPOINT)
    if improved or not BEST_CHECKPOINT.exists():
        temporary_best = RUN_DIR / "best.tmp.pt"
        torch.save(checkpoint, temporary_best)
        temporary_best.replace(BEST_CHECKPOINT)
    print(f"체크포인트 저장: {LAST_CHECKPOINT}")
    if epochs_without_improvement >= EARLY_STOPPING_PATIENCE:
        print(f"Validation Macro F1이 {EARLY_STOPPING_PATIENCE} epoch 동안 개선되지 않아 조기 종료합니다.")
        break

if start_epoch >= EPOCHS:
    _, _, _, val_targets, val_predictions = run_epoch(val_loader, training=False)

print("학습 완료")
print(classification_report(val_targets, val_predictions, target_names=part_names, zero_division=0))

In [ ]:
print(f"Best checkpoint: {BEST_CHECKPOINT} — exists={BEST_CHECKPOINT.is_file()}")
print(f"Last checkpoint: {LAST_CHECKPOINT} — exists={LAST_CHECKPOINT.is_file()}")
print(f"Training history: {HISTORY_CSV} — exists={HISTORY_CSV.is_file()}")
print(f"Dataset: {DATASET_ZIP}")